# FXS / 09 — Previous, 06 signals, 08 signals and everything combined (2020 → 2026-11)

Iván (2026-11-11): the FXS 07 test again, adding the four 2021–2025 finds of FXS 08 (AUDJPY, CADJPY, GBPJPY) — on their own, together with the 06 signals, and all 12 signals together.

| Set | Signals (every trading day, time exits only, news days included) |
|---|---|
| **PREVIOUS** (03b) | EURUSD-02 sell 04:00→08:00 NY · USDJPY-01 sell 00:55→01:55 UTC · USDJPY-02 buy 00:25→00:55 UTC |
| **NEW** (06, 2021–2025 discovery) | N-EU-1 EURUSD sell 07:45→08:00 NY · N-UJ-1 USDJPY sell 00:55→01:10 UTC · N-UJ-2 USDJPY buy 00:25→00:55 UTC (= USDJPY-02) · N-UJ-3 USDJPY buy 15:20→16:20 NY · N-EJ-1 EURJPY buy 14:55→16:55 NY · N-EJ-2 EURJPY sell 06:00→08:00 NY |
| **NEW-08** (08, 2021–2025 discovery, frozen in `08/08_candidates_frozen.csv`) | AUDJPY-N1 buy AUDJPY 12:55→16:55 NY · AUDJPY-N2 buy AUDJPY 01:00→03:00 UTC · CADJPY-N1 buy CADJPY 10:50→14:50 NY · GBPJPY-N1 buy GBPJPY 12:55→16:55 NY |
| **NEW+08** | NEW + NEW-08 (10 signals) |
| **ALL** | PREVIOUS + NEW + NEW-08, with USDJPY-02 counted once (12 signals) |

**What each period means**

| Period | PREVIOUS | NEW (06) |
|---|---|---|
| 2020 | found in 2016–2020 (in sample) | not used to find them (but 2016–2020 was scanned in 06) |
| 2021–2025 | the confirmation years (used to pick the set) | **in sample**: these signals were found here |
| 2026 | seen in FXS 05/06 | seen in FXS 07 |

For **NEW-08**: 2016–2020 was checked in FXS 08 (≈ 0 or negative); 2021–2025 is in sample; **2026 is unseen** — this notebook is the first look. The NEW-08 signals failed the rule stated in FXS 08 (same sign and t ≥ 2 net in 2016–2020), so this is descriptive, not a confirmation test.

Fills at the quoted Dukascopy bid/ask × `SPREAD_FACTOR`, plus `COMMISSION_PER_LOT` per lot round trip. Account: $6,000, size = L × current equity per signal (overlapping signals on the same pair are separate positions).

In [ ]:
import json, os, sys, tempfile
import datetime as dt
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

E0 = 6_000.0
LEVERAGES = [1, 2, 3, 5, 10]
COMMISSION_PER_LOT = 3.0
SPREAD_FACTOR = 0.9
WINDOWS = ((dt.date(2020, 1, 1), dt.date(2020, 12, 31), "discovery"), (dt.date(2021, 1, 1), dt.date(2025, 12, 31), "confirmation"),
           (dt.date(2026, 1, 1), dt.date(2026, 11, 6), "holdout"))
PERIODS = {"2020": (2020, 2020), "2021-2025": (2021, 2025), "2026": (2026, 2026)}
# id: (pair, clock, start slot, end slot, direction); NY slots count minutes from 17:00 New York
SIGNALS = {
    "EURUSD-02": ("EURUSD", "NY", 660, 900, -1), "USDJPY-01": ("USDJPY", "UTC", 55, 115, -1), "USDJPY-02": ("USDJPY", "UTC", 25, 55, 1),
    "N-EU-1": ("EURUSD", "NY", 885, 900, -1), "N-UJ-1": ("USDJPY", "UTC", 55, 70, -1), "N-UJ-3": ("USDJPY", "NY", 1340, 1400, 1),
    "N-EJ-1": ("EURJPY", "NY", 1315, 1435, 1), "N-EJ-2": ("EURJPY", "NY", 780, 900, -1),
    "AUDJPY-N1": ("AUDJPY", "NY", 1195, 1435, 1), "AUDJPY-N2": ("AUDJPY", "UTC", 60, 180, 1), "CADJPY-N1": ("CADJPY", "NY", 1070, 1310, 1),
    "GBPJPY-N1": ("GBPJPY", "NY", 1195, 1435, 1)}
N08 = ["AUDJPY-N1", "AUDJPY-N2", "CADJPY-N1", "GBPJPY-N1"]
SETS = {"PREVIOUS": ["EURUSD-02", "USDJPY-01", "USDJPY-02"],
        "NEW": ["N-EU-1", "N-UJ-1", "USDJPY-02", "N-UJ-3", "N-EJ-1", "N-EJ-2"],
        "NEW-08": N08,
        "NEW+08": ["N-EU-1", "N-UJ-1", "USDJPY-02", "N-UJ-3", "N-EJ-1", "N-EJ-2", *N08],
        "ALL": ["EURUSD-02", "USDJPY-01", "USDJPY-02", "N-EU-1", "N-UJ-1", "N-UJ-3", "N-EJ-1", "N-EJ-2", *N08]}

SYNTHETIC = os.environ.get("FXS_SYNTHETIC") == "1"
if SYNTHETIC:
    os.environ["EQUITY_TRADING_DIR"] = os.environ.get("FXS_SYNTH_DIR") or tempfile.mkdtemp(prefix="fxs09_")
    os.environ.pop("DUKASCOPY_DIR", None)
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "equity_trading").exists())
sys.path.insert(0, str(ROOT / "src"))
from equity_trading import fx_seasonality as F

cfg = F.FXSConfig()
OUT = Path.cwd() / ("09_synthetic" if SYNTHETIC else "09"); OUT.mkdir(exist_ok=True)
cand = pd.DataFrame([{"id": k, "pair": p, "clock": c, "start": s, "end": e, "direction": d} for k, (p, c, s, e, d) in SIGNALS.items()])
cand["rule"] = [f"{'buy' if d > 0 else 'sell'} {r['pair']} {F.slot_label(r['clock'], r['start'])} -> {F.slot_label(r['clock'], r['end']).split()[1]}"
                for (_, r), d in zip(cand.iterrows(), cand["direction"])]
display(cand[["id", "rule"]])
# the 08 signals must match the frozen 08 file exactly
fz = pd.read_csv(Path.cwd() / "08" / "08_candidates_frozen.csv").set_index("id") if not SYNTHETIC else None
if fz is not None:
    for i in N08:
        p_, c_, s_, e_, d_ = SIGNALS[i]
        assert (fz.loc[i, "pair"], fz.loc[i, "clock"], int(fz.loc[i, "start"]), int(fz.loc[i, "end"]), int(fz.loc[i, "direction"])) == (p_, c_, s_, e_, d_), i
pd.set_option("display.width", 220, "display.max_columns", 40)

## 0. Synthetic inputs (smoke run only)

In [ ]:
if SYNTHETIC:
    base = Path(os.environ["EQUITY_TRADING_DIR"]) / "Dukascopy"
    for i, p in enumerate(("EURUSD", "USDJPY", "EURJPY", "AUDJPY", "CADJPY", "GBPJPY")):
        F.write_synthetic_candles(base, p, dt.date(2019, 12, 28), dt.date(2026, 11, 6), seed=41 + i, drift_bps=-3.0 if p == "EURUSD" else 0.0,
                                  drift_ny=(4, 8), price=150.0 if p.endswith("JPY") else 1.10)

## 1. Daily results per signal

In [ ]:
for p in sorted(set(cand["pair"])):
    for a, b, _ in WINDOWS:
        cov = F.file_coverage(p, a, b)
        assert cov["share"] >= cfg.min_file_share, f"{p} {a.year}: only {cov['share']:.1%} of day files"
parts = [F.structure_detail(cand, a, b, st, cfg, with_price=True, spread_factor=SPREAD_FACTOR) for a, b, st in WINDOWS]
G = pd.concat([x[0] for x in parts]).sort_index().fillna(0.0)
M = pd.concat([x[1] for x in parts]).sort_index().fillna(0.0)
P = pd.concat([x[2] for x in parts]).sort_index().ffill().bfill()
PAIR_OF = cand.set_index("id")["pair"]
comm = pd.Series({i: F.commission_bps_per_lot(PAIR_OF[i], COMMISSION_PER_LOT) for i in G.columns})
N = G - comm
traded = G != 0
pips = pd.DataFrame({i: F.bps_to_pips(N[i], P[i], PAIR_OF[i]) for i in G.columns}, index=G.index).where(traded)
usd = pd.DataFrame({i: pips[i] * F.usd_per_pip_per_lot(PAIR_OF[i], P[i]) for i in G.columns}, index=G.index)
year = pips.index.year
rows = []
for i in cand["id"]:
    r = {"signal": i, "rule": cand.set_index("id").loc[i, "rule"]}
    for per, (a, b) in PERIODS.items():
        x = pips[i][(year >= a) & (year <= b)].dropna()
        r[f"{per} net pips"] = x.mean(); r[f"{per} t"] = x.mean() / (x.std() / np.sqrt(len(x))) if len(x) > 2 else np.nan
    x = pips[i].dropna()
    r.update({"all net pips": x.mean(), "win rate": (x > 0).mean(), "USD per 1-lot trade": usd[i].mean(), "swing pips": x.std()})
    rows.append(r)
sig = pd.DataFrame(rows).set_index("signal")
display(sig.round(2)); sig.to_csv(OUT / "09_signals.csv")
yr = pips.groupby(year).mean()
display(yr.round(2)); yr.to_csv(OUT / "09_signals_yearly_pips.csv")

## 2. How the sets move together

Daily USD result per set at 1 lot per signal, and the correlation between PREVIOUS and NEW.

In [ ]:
day_usd = {s: usd[ids].fillna(0).sum(axis=1) for s, ids in SETS.items()}
dd = pd.DataFrame(day_usd)
print("correlation of daily USD results between sets:"); display(dd.corr().round(2))
display(pd.DataFrame({s: {"mean USD/day (1 lot each)": dd[s].mean(), "st.dev USD/day": dd[s].std(),
                          "mean / st.dev": dd[s].mean() / dd[s].std(), "worst day USD": dd[s].min()} for s in SETS}).T.round(2))
display(pd.DataFrame(usd).fillna(0).corr().round(2))

## 3. The $6,000 compounding account, 2020 → 2026

In [ ]:
rows, curves = [], {}
for s, ids in SETS.items():
    for L in LEVERAGES:
        acc = F.simulate_account(N[ids], pd.Series(float(L), index=ids), E0)
        curves[(s, L)] = acc
        prev = acc["equity"].shift(1).fillna(E0).to_numpy()
        units = np.floor(prev[:, None] * L / 1000) * 1000
        worst = ((units * (M[ids].to_numpy() - comm[ids].to_numpy())).sum(axis=1) / 1e4 / np.maximum(prev, 1e-9)).min()
        st = F.account_stats(acc, E0)
        rows.append({"set": s, "signals": len(ids), "L each": L, "final_equity": round(st["final_equity"]), "cagr %": round(100 * st["cagr"], 1),
                     "sharpe": round(st["sharpe"], 2), "max drawdown %": round(100 * st["max_drawdown"], 1),
                     "worst year %": round(100 * st["worst_year"], 1), "worst day % (all open at worst)": round(100 * worst, 1)})
acc_tab = pd.DataFrame(rows)
display(acc_tab); acc_tab.to_csv(OUT / "09_account.csv", index=False)
fig, axes = plt.subplots(1, len(SETS), figsize=(24, 4.2), sharey=True)
for ax, s in zip(axes, SETS):
    for L in LEVERAGES:
        ax.plot(curves[(s, L)]["equity"].clip(lower=1), label=f"{L}x each")
    ax.axhline(E0, color="grey", lw=0.6)
    for b_ in ("2021-01-01", "2026-01-01"):
        ax.axvline(pd.Timestamp(b_), color="black", ls=":", lw=0.8)
    ax.set_yscale("log"); ax.set_title(f"{s} ({len(SETS[s])} signals)"); ax.legend()
axes[0].set_ylabel("equity USD (log)")
plt.tight_layout(); plt.savefig(OUT / "09_equity.png", dpi=110); plt.show()

## 4. Each period on its own (separate $6,000 accounts)

In [ ]:
pr = []
for s, ids in SETS.items():
    for per, (a, b) in PERIODS.items():
        sel = (N.index.year >= a) & (N.index.year <= b)
        for L in LEVERAGES:
            st = F.account_stats(F.simulate_account(N.loc[sel, ids], pd.Series(float(L), index=ids), E0), E0)
            pr.append({"set": s, "period": per, "L": L, "cagr %": round(100 * st["cagr"], 1), "max dd %": round(100 * st["max_drawdown"], 1)})
pr = pd.DataFrame(pr)
display(pr.pivot_table(index=["set", "L"], columns="period", values=["cagr %", "max dd %"])[[("cagr %", p) for p in PERIODS] + [("max dd %", p) for p in PERIODS]])
pr.to_csv(OUT / "09_period_account.csv", index=False)

## 5. Year by year (account return %, 2× and 10× each)

In [ ]:
for L in (2, 10):
    yrs = {}
    for s in SETS:
        e = curves[(s, L)]["equity"].resample("YE").last()
        r = e.pct_change(); r.iloc[0] = e.iloc[0] / E0 - 1; r.index = r.index.year
        yrs[s] = 100 * r
    yrs = pd.DataFrame(yrs)
    print(f"{L}x each:"); display(yrs.round(1)); yrs.to_csv(OUT / f"09_yearly_returns_{L}x.csv")

In [ ]:
summary = {"study": "FXS09", "descriptive": True, "synthetic": SYNTHETIC, "signals": {k: list(v) for k, v in SIGNALS.items()}, "sets": SETS,
           "commission_usd_per_lot_rt": COMMISSION_PER_LOT, "spread_factor": SPREAD_FACTOR, "equity0": E0,
           "signals_table": sig.round(4).reset_index().to_dict(orient="records"), "account": acc_tab.to_dict(orient="records"),
           "note": "NEW (06) and NEW-08 found on 2021-2025 (in sample there); 2026 first look for NEW-08; NEW-08 failed the FXS 08 2016-2020 rule."}
(OUT / "summary.json").write_text(json.dumps(summary, indent=2, default=str))
print("written", OUT / "summary.json")